# dataloader-batching — faded example 1: Build train and eval loaders with the right shuffle flags

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-batching`. The last cell reports your progress on the `PyTorch: DataLoader batching` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader batching` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-batching`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-batching"
DD_SUBTOPIC = "PyTorch: DataLoader batching"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The ARENA-canonical convention is `shuffle=True` for the training loader (so gradient batches differ each epoch) and `shuffle=False` for the evaluation loader (so metrics are reproducible). Both wrap a `TensorDataset` of aligned feature/label tensors. The partial last batch is kept by default (`drop_last=False`).

## Faded exercise 1

### Faded — build a train loader and an eval loader

Implement `build_loaders(x_tr, y_tr, x_ev, y_ev, batch_size)`. Wrap each split in a `TensorDataset`, then construct a training `DataLoader` with shuffling ON and an eval `DataLoader` with shuffling OFF, both with the given `batch_size`. Return `(train_loader, eval_loader)`.

The `TensorDataset` wrapping and the eval loader are already written for you. **Complete the one line that constructs the training loader with the correct shuffle flag.**

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def build_loaders(x_tr, y_tr, x_ev, y_ev, batch_size):
    train_ds = TensorDataset(x_tr, y_tr)
    eval_ds = TensorDataset(x_ev, y_ev)
    train_loader = None  # TODO: fill in this step — read the prompt cell above
    eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)
    return train_loader, eval_loader


def _test():
    from torch.utils.data import DataLoader
    t.manual_seed(0)
    x_tr = t.randn(30, 3)
    y_tr = t.randint(0, 2, (30,))
    x_ev = t.randn(12, 3)
    y_ev = t.randint(0, 2, (12,))
    train_loader, eval_loader = build_loaders(x_tr, y_tr, x_ev, y_ev, 8)
    assert isinstance(train_loader, DataLoader)
    assert isinstance(eval_loader, DataLoader)
    # eval is reproducible (shuffle off)
    o1 = t.cat([ids for ids, _ in eval_loader])
    o2 = t.cat([ids for ids, _ in eval_loader])
    assert t.equal(o1, o2), 'eval loader must be reproducible'
    # train is shuffled: order should change across epochs
    s1 = t.cat([yb for _, yb in train_loader])
    # full coverage each epoch
    assert s1.numel() == 30
    g1 = t.cat([xb for xb, _ in train_loader])
    g2 = t.cat([xb for xb, _ in train_loader])
    assert not t.equal(g1, g2), 'train loader must reshuffle across epochs'
    # batch shapes
    xb, yb = next(iter(train_loader))
    assert xb.shape == (8, 3) and yb.shape == (8,)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch.utils.data import TensorDataset, DataLoader


def build_loaders(x_tr, y_tr, x_ev, y_ev, batch_size):
    train_ds = TensorDataset(x_tr, y_tr)
    eval_ds = TensorDataset(x_ev, y_ev)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)
    return train_loader, eval_loader
```
</details>